# ICLR End-to-End Pipeline (Notebook)

This notebook mirrors `iclr.py` and walks through the full experimental pipeline.
Each section explains what it does and produces the same artifacts as the CLI sequence.


## 0) Configuration
Set paths and core parameters. These mirror the CLI flags in `iclr.py`.


In [ ]:
from pathlib import Path

# Base output directory
BASE_OUT = Path("pyident_results")

# Subdirectories (match iclr.py defaults)
ENSEMBLE_DIR = BASE_OUT / "fresh_ensemble"
X0_DIR = BASE_OUT / "fresh_ABx0"
PBH_DIR = BASE_OUT / "fresh_nonidentifiable_ABx0"
BOXPLOT_DIR = PBH_DIR / "boxplots"

# Sweep parameters
SPARSITY_GRID = "0.0:0.1:1.0"
NDIM_GRID = "2:1:10"
SAMPLES_PER_CELL = 10000

# Density filter
DENSITY_MIN = 0.3
DENSITY_MAX = 0.7
DENSITY_SOURCE = "AB"

# x0 sampling for score plots
X0_SAMPLES = 10
MASK_PS = [0.25, 0.5, 0.75]
OUTLIER_TRIM = 0.05

# PBH selection + estimation
MASK_PS_PBH = [0.25, 0.5, 0.75, 1.0]
PBH_THRESHOLD = 1e-6
SEED = 12345
T = 100
DT = 1.0
U_SCALE = 3.0
DWELL = 1
ALGOS = "DMDc"  # or e.g. "SINDy,DMDc,MOESP,NODE"


## 1) Sweep systems + save matrices
Runs `sim_regcomb_ctrb` across a sparsity × state-dimension grid.
Outputs: `scores_summary.csv`, `systems.csv`, `systems_matrices.npz`.


In [ ]:
from pyident.experiments import sim_regcomb_ctrb

args = sim_regcomb_ctrb.build_parser().parse_args([
    "--axes", "sparsity,ndim",
    "--sparsity-grid", SPARSITY_GRID,
    "--ndim-grid", NDIM_GRID,
    "--samples", str(SAMPLES_PER_CELL),
    "--outdir", str(ENSEMBLE_DIR),
    "--save-matrices",
])
if args.m is None:
    args.m = int(args.n)
sim_regcomb_ctrb.run(args)


## 2) Filter uncontrollable systems by density
Reads the ensemble and keeps only uncontrollable systems with density in [0.3, 0.7].
Outputs: `systems_unctrb_d0.3_0.7.csv/.npz`.


In [ ]:
from pyident.experiments import filter_unctrb_dataset

args = filter_unctrb_dataset.build_parser().parse_args([
    "--outdir", str(ENSEMBLE_DIR),
    "--density-min", str(DENSITY_MIN),
    "--density-max", str(DENSITY_MAX),
    "--density-source", DENSITY_SOURCE,
])
filter_unctrb_dataset.run(args)


## 3) Identifiability score boxplots vs x0 sampling
For the filtered (A,B) pool, sample x0 under different sparsity masks
and compute PBH / left-eigenvector scores.
Outputs: `identifiability_scores.csv` and boxplots.


In [ ]:
from pyident.experiments import sim_unctrb_x0_boxplot

filtered_csv = ENSEMBLE_DIR / f"systems_unctrb_d{DENSITY_MIN:g}_{DENSITY_MAX:g}.csv"
filtered_npz = ENSEMBLE_DIR / f"systems_unctrb_d{DENSITY_MIN:g}_{DENSITY_MAX:g}.npz"

args = sim_unctrb_x0_boxplot.build_parser().parse_args([
    "--dataset-csv", str(filtered_csv),
    "--dataset-npz", str(filtered_npz),
    "--x0-samples", str(X0_SAMPLES),
    "--mask-ps", *[str(p) for p in MASK_PS],
    "--mask-renorm",
    "--outdir", str(X0_DIR),
    "--outlier-trim", str(OUTLIER_TRIM),
])
sim_unctrb_x0_boxplot.run(args)


## 4) Select low-PBH (A,B,x0) triples + estimate (A,B)
Re-samples x0 (same policy) and keeps only triples with PBH < threshold.
Simulates PRBS trajectories and runs estimators.
Outputs: `selected_*.csv/.npz` and `estimation_errors.csv`.


In [ ]:
from pyident.experiments import sim_unctrb_pbh_estimators

args = sim_unctrb_pbh_estimators.build_parser().parse_args([
    "--dataset-csv", str(filtered_csv),
    "--dataset-npz", str(filtered_npz),
    "--outdir", str(PBH_DIR),
    "--seed", str(SEED),
    "--x0-samples", str(X0_SAMPLES),
    "--mask-ps", *[str(p) for p in MASK_PS_PBH],
    "--mask-renorm",
    "--pbh-threshold", str(PBH_THRESHOLD),
    "--T", str(T),
    "--dt", str(DT),
    "--u-scale", str(U_SCALE),
    "--dwell", str(DWELL),
    "--algos", ALGOS,
])
sim_unctrb_pbh_estimators.run(args)


## 5) Error boxplots (standard vs P-basis)
Loads the selected triples and re-runs estimators to plot error distributions.
Outputs: `boxplot_err_standard.png`, `boxplot_err_Pbasis.png`, `boxplot_err_standard_vs_Pbasis.png`.


In [ ]:
from pyident.experiments import sim_unctrb_pbh_error_boxplots

selected_suffix = f"pbh_lt_{PBH_THRESHOLD:g}"
selected_csv = PBH_DIR / f"selected_{selected_suffix}.csv"
selected_npz = PBH_DIR / f"selected_{selected_suffix}.npz"

args = sim_unctrb_pbh_error_boxplots.build_parser().parse_args([
    "--selected-npz", str(selected_npz),
    "--selected-csv", str(selected_csv),
    "--outdir", str(BOXPLOT_DIR),
    "--seed", str(SEED),
    "--T", str(T),
    "--dt", str(DT),
    "--u-scale", str(U_SCALE),
    "--dwell", str(DWELL),
    "--algos", ALGOS,
])
sim_unctrb_pbh_error_boxplots.run(args)


---
## Notes
- This notebook runs the full pipeline in-process (no shell commands).
- The outputs mirror the CLI sequence.
- If you want log-scale plots, pass `--yscale log` in the plotting steps.
